# 02 - Limpieza de datos

Cargar los CSV crudos de `data/raw/`, los unifica, limpia del texto, detexta idioma y exporta dos a `data/processed/`

## 1. Imports y configuración

In [35]:
import pandas as pd
import numpy as np
import re
import unicodedata
from   langdetect import detect, LangDetectException
import glob
import os

pd.set_option('display.max_colwidth',150)  # para ver texto completo al isnpeccionar

# 2. Cargar y unificar los CSV de todos los restaurantes

In [ ]:
#Listado de todos los archivos en data/raw
archivos = glob.glob('../data/raw/*.csv')
print(f'Archivos encontrados: {len(archivos)}')

for a in archivos:
    print(f' -', a)

Archivos encontrados: 8
 - ../data/raw\casa_res.csv
 - ../data/raw\don_parrilla_steak_house.csv
 - ../data/raw\el_dorado.csv
 - ../data/raw\la_casa_del_Tomahawk_reviews_raw.csv
 - ../data/raw\morogrill_reviews_raw.csv
 - ../data/raw\parrillada_punta_del_este_reviews_raw.csv
 - ../data/raw\parrilla_del_ñato_reviews_raw.csv
 - ../data/raw\rukito_reviews_raw.csv


In [20]:
# Carga de achivos a df
columnas_utiles = [
    'text', 'stars', 'publishedAtDate', 'title',
    'reviewDetailedRating/Comida', 'reviewDetailedRating/Servicio', 
    'reviewDetailedRating/Ambiente', 'reviewContext/Tiempo de espera',
    'likesCount', 'reviewerNumberOfReviews', 'isLocalGuide'
]

dfs = []
for archivo in archivos: 
    df_temp = pd.read_csv(archivo)
    df_temp = df_temp[columnas_utiles] #Nos quedamos con las columnas útiles
    dfs.append(df_temp)
    
df =pd.concat(dfs, ignore_index=True)
print(f'Total de reseñas combinadas: {len(df)}')
df.head()

Total de reseñas combinadas: 2372


,text,stars,publishedAtDate,title,reviewDetailedRating/Comida,reviewDetailedRating/Servicio,reviewDetailedRating/Ambiente,reviewContext/Tiempo de espera,likesCount,reviewerNumberOfReviews,isLocalGuide
0,"La comida muy buena, pero el servicio estuvo deficiente, demoraban demasiado, tuve que pedirle a 3 personas lo mismo para que lo hagan llegar, con...",3,2026-08-07T19:57:17.274Z,Casa Res | Steak House (Francisco de Orellana),5.0,3.0,2.0,NaN,0,0,False
1,Comida fresca y ensaladas para elegir.,5,2026-08-07T17:05:31.989Z,Casa Res | Steak House (Francisco de Orellana),5.0,5.0,5.0,Sin espera,0,68,True
2,Servicio lento,3,2026-08-04T21:51:27.788Z,Casa Res | Steak House (Francisco de Orellana),NaN,NaN,NaN,NaN,0,37,True
3,NaN,5,2026-08-03T01:04:14.609Z,Casa Res | Steak House (Francisco de Orellana),5.0,5.0,5.0,NaN,0,195,True
4,NaN,5,2026-08-01T17:08:39.545Z,Casa Res | Steak House (Francisco de Orellana),5.0,5.0,5.0,NaN,0,0,False


In [23]:
print(f'shape:  {df.shape}')
print("\nRestaurantes cargados")
print(df['title'].value_counts())
print("\nNulos por columna")
print(df.isnull().sum())


shape:  (2372, 11)

Restaurantes cargados
title
Casa Res | Steak House (Francisco de Orellana)    300
Parrillada Restaurant El Dorado Sauces 3          300
La Casa del Tomahawk                              300
MoroGrill - Urdesa                                300
Parrillada Punta Del Este                         300
La Parrilla Del Ñato - Urdesa                     300
Rukito Grill&Drink - Alborada                     300
Don Parrilla Steak House - Urdesa                 272
Name: count, dtype: int64

Nulos por columna
text                              1285
stars                                0
publishedAtDate                      0
title                                0
reviewDetailedRating/Comida        707
reviewDetailedRating/Servicio      709
reviewDetailedRating/Ambiente      718
reviewContext/Tiempo de espera    1900
likesCount                           0
reviewerNumberOfReviews              0
isLocalGuide                         0
dtype: int64


In [28]:
# Separamos las reseñas con texto vs solo rating
df['tiene_texto'] = df['text'].notna()

print(f'Con texto: {df['tiene_texto'].sum()}')
print(f'Solo rating: {(~df['tiene_texto']).sum()}')

# Dataset compelto (para métricas de rating general)
df_completo = df.copy()

#Dataset solo con texto (para NLP -sentimiento y temas)
df_texto = df[df['tiene_texto']].copy()

Con texto: 1087
Solo rating: 1285


In [30]:
def limpiar_texto(texto): 
    if pd.isna(texto): 
        return texto
    
    # Normalizar unicode (maneja tildes/ñ correctamente)
    texto = unicodedata.normalize('NFC',texto)
    
    #Quitar saltos de líneas múltiples y espacios extra
    texto = re.sub(r'\s+',' ',texto)
    
    #Quitar espacioes al inicio/final
    texto = texto.strip()
    
    return texto

df_texto['text_limpio'] = df_texto['text'].apply(limpiar_texto)

In [49]:
#Detección de idioma
def detectar_idioma_seguro(texto): 
    if pd.isna(texto) or len(texto.strip()) < 20: 
        return 'es'
    try: 
        return detect(texto)
    except LangDetectException:
        return 'desconocido'
    
df_texto['idioma_detectado'] = df_texto['text_limpio'].apply(detectar_idioma_seguro)
print(df_texto['idioma_detectado'].value_counts())

idioma_detectado
es             1039
en               15
pt               15
ca                6
it                3
ro                3
de                2
vi                1
desconocido       1
ko                1
zh-cn             1
Name: count, dtype: int64


In [ ]:
# Parsear fechas
df_completo['fecha'] = pd.to_datetime(df_completo['publishedAtDate'])
df_texto['fecha'] = pd.to_datetime(df_texto['publishedAtDate'])

print(f'Reseñas desde: {df_completo['fecha'].min()}')
print(f'Reseñas desde: {df_completo['fecha'].max()}')

Reseñas desde: 2021-02-20 22:29:55.743000+00:00
Reseñas desde: 2026-08-08 13:54:14.417000+00:00


In [55]:
df_texto = df_texto.rename(columns={
    'title': 'restaurante',
    'text_limpio': 'review_texto',
    'stars': 'rating',
    'reviewDetailedRating/Comida':'rating_comida',
    'reviewDetailedRating/Servicio':'rating_servicio',
    'reviewDetailedRating/Ambiente':'rating_ambiente',
    'reviewContext/Tiempo de espera':'tiempo_espera_reportado',
})

df_texto.head(5)

,text,rating,publishedAtDate,restaurante,rating_comida,rating_servicio,rating_ambiente,tiempo_espera_reportado,likesCount,reviewerNumberOfReviews,isLocalGuide,tiene_texto,review_texto,idioma_detectado,fecha
0,"La comida muy buena, pero el servicio estuvo deficiente, demoraban demasiado, tuve que pedirle a 3 personas lo mismo para que lo hagan llegar, con...",3,2026-08-07T19:57:17.274Z,Casa Res | Steak House (Francisco de Orellana),5.0,3.0,2.0,NaN,0,0,False,True,"La comida muy buena, pero el servicio estuvo deficiente, demoraban demasiado, tuve que pedirle a 3 personas lo mismo para que lo hagan llegar, con...",es,2026-08-07 19:57:17.274000+00:00
1,Comida fresca y ensaladas para elegir.,5,2026-08-07T17:05:31.989Z,Casa Res | Steak House (Francisco de Orellana),5.0,5.0,5.0,Sin espera,0,68,True,True,Comida fresca y ensaladas para elegir.,es,2026-08-07 17:05:31.989000+00:00
2,Servicio lento,3,2026-08-04T21:51:27.788Z,Casa Res | Steak House (Francisco de Orellana),NaN,NaN,NaN,NaN,0,37,True,True,Servicio lento,es,2026-08-04 21:51:27.788000+00:00
7,"Pedí una picaña de res y el corte me llegó frío, lo que evidencia que el plato estuvo listo mucho antes de que me lo entregaran; y por si fuera po...",1,2026-07-26T19:57:56.005Z,Casa Res | Steak House (Francisco de Orellana),2.0,1.0,1.0,NaN,0,1,False,True,"Pedí una picaña de res y el corte me llegó frío, lo que evidencia que el plato estuvo listo mucho antes de que me lo entregaran; y por si fuera po...",es,2026-07-26 19:57:56.005000+00:00
8,"Hicimos una reserva para 24 personas por un cumpleaños, nos pusieron en una parte donde no funcionaba el aire, los cortes en terminos no solicitad...",1,2026-07-26T16:14:14.191Z,Casa Res | Steak House (Francisco de Orellana),1.0,1.0,1.0,Sin espera,0,1,False,True,"Hicimos una reserva para 24 personas por un cumpleaños, nos pusieron en una parte donde no funcionaba el aire, los cortes en terminos no solicitad...",es,2026-07-26 16:14:14.191000+00:00


In [57]:
df_completo.to_csv('../data/processed/reviews_all.csv', index=False)
df_texto.to_csv('../data/processed/reviews_with_text.csv',index=False)

print('Archivos Guardados en data/processed/')

Archivos Guardados en data/processed/
